# 16 — MARS Case Studies: Interpretability Through Individual Learner Trajectories

**Goal:** Demonstrate that the multi-agent architecture of MARS produces *interpretable*
recommendations by tracing the full decision pipeline for 5 representative students
(one per cluster).

For each student we show:
1. **Profile** — IRT ability (theta), cluster assignment, interaction volume, accuracy, dominant part
2. **Timeline** — 50 steps of MARS recommendations with per-step agent outputs
3. **Visualisation** — theta trajectory, confidence class distribution, strategy weights, accuracy breakdown
4. **Interpretation** — WHY MARS chose each recommendation (gap -> prerequisite -> lecture chain)
5. **Thompson Sampling dynamics** — how KB/CB/CF strategy weights evolve over time

This is a strong argument for multi-agent vs monolithic architecture:
every recommendation can be *explained* by pointing to the specific agent outputs
that contributed to it.

In [ ]:
"""Cell 1 — Imports and setup."""

from __future__ import annotations

import json
import logging
import sys
import time
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from agents.utils import set_global_seed, load_config

logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger("case_studies")
logger.setLevel(logging.INFO)

SEED = 42
N_STEPS = 50
CONTEXT_WINDOW = 10        # initial context interactions
GT_WINDOW = 10             # ground-truth look-ahead window
MIN_TEST_INTERACTIONS = 60 # minimum test interactions to be eligible
N_CASE_STUDENTS = 5        # one per cluster

RESULTS_DIR = ROOT / "results" / "case_studies"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CONFIG = load_config(str(ROOT / "configs" / "config.yaml"))

set_global_seed(SEED)

# Plotting style
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
})

STRATEGY_COLORS = {
    "knowledge_based": "#1f77b4",
    "content_based":   "#ff7f0e",
    "collaborative":   "#2ca02c",
}
STRATEGY_LABELS = {
    "knowledge_based": "Knowledge-Based",
    "content_based":   "Content-Based",
    "collaborative":   "Collaborative",
}

print(f"Project root: {ROOT}")
print(f"Results dir:  {RESULTS_DIR}")
print(f"N steps:      {N_STEPS}")

In [ ]:
"""Cell 2 — Load data and train all agents."""

from data.loader import EdNetLoader
from agents.diagnostic_agent import DiagnosticAgent
from agents.confidence_agent import ConfidenceAgent
from agents.kg_agent import KnowledgeGraphAgent
from agents.prediction_agent import PredictionAgent
from agents.recommendation_agent import RecommendationAgent
from agents.personalization_agent import (
    PersonalizationAgent, extract_user_features, FEATURE_NAMES,
)
from agents.orchestrator import Orchestrator
from scripts.run_multi_seed import train_all_agents

# Load splits
splits_dir = ROOT / "data" / "splits"
train_df = pd.read_parquet(splits_dir / "train.parquet")
val_df = pd.read_parquet(splits_dir / "val.parquet")
test_df = pd.read_parquet(splits_dir / "test.parquet")

loader = EdNetLoader(data_dir=str(ROOT / "data" / "raw"))
questions_df = loader.questions
lectures_df = loader.lectures

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")
print(f"Test users: {test_df['user_id'].nunique()}")
print(f"Questions: {len(questions_df)}, Lectures: {len(lectures_df)}")

In [ ]:
"""Cell 3 — Train agents and create orchestrator."""

agents, agent_metrics = train_all_agents(train_df, val_df, seed=SEED, config=CONFIG)

diag = agents["diagnostic"]
conf = agents["confidence"]
kg   = agents["knowledge_graph"]
pred = agents["prediction"]
rec  = agents["recommendation"]

# Personalization: cluster assignment
pers = PersonalizationAgent()
user_feats = extract_user_features(train_df)
optimal_k = pers.train_clusters(user_feats)

# Create orchestrator
orch = Orchestrator(seed=SEED)
for agent in agents.values():
    orch.register_agent(agent)
orch.register_agent(pers)

print(f"\nOptimal K = {optimal_k}")
print(f"Cluster names: {pers._cluster_names}")
print(f"Agent metrics: {list(agent_metrics.keys())}")

In [ ]:
"""Cell 4 — Assign clusters to test users and select one per cluster."""

# Extract features for test users from their train history
# (cluster assignment is based on train behaviour)
test_user_ids = test_df["user_id"].unique()
test_user_counts = test_df.groupby("user_id").size()
eligible_users = test_user_counts[test_user_counts >= MIN_TEST_INTERACTIONS].index.tolist()
print(f"Test users with >= {MIN_TEST_INTERACTIONS} interactions: {len(eligible_users)}")

# Assign clusters to eligible users
user_cluster_map = {}  # user_id -> cluster_id
user_profiles = {}     # user_id -> cluster profile dict

for uid in eligible_users:
    uid_str = str(uid)
    # Check if user has train data for feature extraction
    user_train = train_df[train_df["user_id"] == uid]
    if len(user_train) >= 20:
        profile = pers.assign_cluster(uid_str, interactions=user_train)
    else:
        profile = pers.assign_cluster(uid_str)
    user_cluster_map[uid] = profile["cluster_id"]
    user_profiles[uid] = profile

print(f"Assigned clusters to {len(user_cluster_map)} users")
cluster_counts = pd.Series(user_cluster_map).value_counts().sort_index()
for cid, count in cluster_counts.items():
    print(f"  Cluster {cid} ({pers._cluster_names.get(cid, '?')}): {count} users")

In [ ]:
"""Cell 5 — Select one representative student per cluster.

Criteria: most test interactions among users closest to cluster centroid.
"""

selected_students = {}  # cluster_id -> user_id

for cid in range(optimal_k):
    # Users in this cluster
    cluster_users = [uid for uid, c in user_cluster_map.items() if c == cid]
    if not cluster_users:
        print(f"WARNING: No eligible users in cluster {cid}")
        continue

    # Rank by closeness to centroid (lower distance = more representative),
    # then by number of test interactions (more = richer case study)
    candidates = []
    for uid in cluster_users:
        dist = user_profiles[uid].get("distance_to_centroid", 999)
        n_test = int(test_user_counts.get(uid, 0))
        candidates.append((uid, dist, n_test))

    # Sort by distance (ascending), then n_test (descending)
    candidates.sort(key=lambda x: (x[1], -x[2]))

    # Pick the best candidate
    best_uid = candidates[0][0]
    selected_students[cid] = best_uid
    p = user_profiles[best_uid]
    print(
        f"Cluster {cid} ({pers._cluster_names.get(cid, '?')}): "
        f"user={best_uid}, dist={p['distance_to_centroid']:.3f}, "
        f"n_test={int(test_user_counts.get(best_uid, 0))}"
    )

print(f"\nSelected {len(selected_students)} students for case studies.")

In [ ]:
"""Cell 6 — Helper: parse tags from interactions."""

NUM_TAGS = 293


def parse_tags(tags):
    """Parse tags from various formats to a list of ints."""
    if isinstance(tags, list):
        return [int(t) for t in tags if 0 <= int(t) < NUM_TAGS]
    if isinstance(tags, (int, np.integer)):
        return [int(tags)] if 0 <= int(tags) < NUM_TAGS else []
    if isinstance(tags, str):
        return [
            int(t)
            for t in tags.split(";")
            if t.strip().isdigit() and 0 <= int(t) < NUM_TAGS
        ]
    return []


def get_part_name(part_id):
    """TOEIC part names."""
    names = {
        1: "Photographs",
        2: "Q&A",
        3: "Conversations",
        4: "Talks",
        5: "Incomplete Sent.",
        6: "Text Completion",
        7: "Reading Comp.",
    }
    return names.get(int(part_id), f"Part {part_id}")


print("Helpers ready.")

In [ ]:
"""Cell 7 — Simulate 50-step MARS pipeline for each selected student.

At each step we record:
  - theta (IRT ability)
  - confidence class distribution
  - strategy selected + TS weights
  - recommended items
  - gap tags -> prerequisite chain -> lecture mapping
"""


def simulate_student(uid, test_interactions, n_steps=N_STEPS):
    """Run MARS pipeline step-by-step for a single student.

    Returns a list of dicts, one per step, with full agent outputs.
    """
    uid_str = str(uid)
    interactions = test_interactions.sort_values("timestamp").reset_index(drop=True)

    if len(interactions) < n_steps + CONTEXT_WINDOW:
        logger.warning("User %s: only %d interactions, need %d",
                       uid, len(interactions), n_steps + CONTEXT_WINDOW)
        n_steps = len(interactions) - CONTEXT_WINDOW
        if n_steps <= 0:
            return []

    steps = []

    for step in range(n_steps):
        ctx_end = CONTEXT_WINDOW + step
        ctx = interactions.iloc[:ctx_end]

        # --- 1. IRT assessment ---
        try:
            diag_result = diag.run_assessment(uid_str, ctx)
            theta = diag_result.get("theta", 0.0)
            theta_se = diag_result.get("se", 1.0)
        except Exception:
            theta, theta_se = 0.0, 1.0
            diag_result = {"theta": theta, "se": theta_se}

        # --- 2. Confidence classification ---
        try:
            conf_result = conf.classify_batch(uid_str, ctx)
            conf_classes = conf_result.get("classes", [])
            conf_names = conf_result.get("class_names", [])
        except Exception:
            conf_classes, conf_names = [], []
            conf_result = {"classes": [], "class_names": []}

        # --- 3. KG profile update ---
        try:
            kg_result = kg.update_user_profile(uid_str, diag_result, conf_result)
            gap_tags = kg_result.get("gap_tags", [])
            prereq_gaps = kg_result.get("prerequisite_gaps", {})
            kg_recs = kg_result.get("recommendations", [])
        except Exception:
            gap_tags, prereq_gaps, kg_recs = [], {}, []
            kg_result = {}

        # --- 4. TS strategy selection + weights ---
        n_interactions = len(ctx)
        try:
            strategy = rec.select_strategy(uid_str, n_interactions=n_interactions)
        except Exception:
            strategy = "knowledge_based"

        try:
            ts_weights = rec.get_ts_weights(uid_str)
        except Exception:
            ts_weights = {"knowledge_based": 0.9, "content_based": 0.8, "collaborative": 0.5}

        # --- 5. Recommendation ---
        try:
            rec_result = rec.recommend(
                uid_str, kg_profile=kg_result, confidence=conf_result, n=10,
            )
            rec_items = rec_result.get("items", [])
            if not rec_items:
                rec_items = rec_result.get("recommendations", [])
        except Exception:
            rec_items = []
            rec_result = {}

        # --- 6. Compute ground-truth accuracy for this step ---
        current_row = interactions.iloc[ctx_end - 1] if ctx_end > 0 else {}
        correct = bool(current_row.get("correct", False)) if isinstance(current_row, (dict, pd.Series)) else False
        part_id = int(current_row.get("part_id", 1)) if isinstance(current_row, (dict, pd.Series)) else 1
        current_tags = parse_tags(current_row.get("tags", []))

        # Running accuracy per part
        if "part_id" in ctx.columns:
            part_acc = ctx.groupby("part_id")["correct"].mean().to_dict()
        else:
            part_acc = {}

        # --- 7. Compute reward: update TS posterior ---
        gt_end = min(ctx_end + GT_WINDOW, len(interactions))
        future = interactions.iloc[ctx_end:gt_end]
        gt_fail_tags = set()
        for _, row in future.iterrows():
            if not row["correct"]:
                gt_fail_tags.update(parse_tags(row.get("tags", [])))

        # Compute reward for TS update
        rec_tag_set = set()
        for r in rec_items:
            if isinstance(r, dict):
                rec_tag_set.update(r.get("related_tags", []))
        if gt_fail_tags:
            reward = len(rec_tag_set & gt_fail_tags) / max(len(gt_fail_tags), 1)
        else:
            reward = 0.5  # neutral when no gaps

        try:
            rec.update_reward(uid_str, strategy, reward)
        except Exception:
            pass

        # --- Record step ---
        steps.append({
            "step": step,
            "theta": float(theta),
            "theta_se": float(theta_se),
            "conf_classes": list(conf_classes[-5:]) if conf_classes else [],
            "conf_names": list(conf_names[-5:]) if conf_names else [],
            "conf_distribution": (
                pd.Series(conf_names).value_counts(normalize=True).to_dict()
                if conf_names else {}
            ),
            "strategy": strategy,
            "ts_weights": dict(ts_weights),
            "gap_tags": list(gap_tags[:10]),
            "prerequisite_gaps": {
                str(k): v[:3] for k, v in list(prereq_gaps.items())[:5]
            },
            "recommended_items": [
                {
                    "item_id": r.get("item_id", r.get("question_id", "?")),
                    "item_type": r.get("item_type", "question"),
                    "reason": r.get("reason", ""),
                    "related_tags": r.get("related_tags", []),
                }
                if isinstance(r, dict)
                else {"item_id": str(r), "item_type": "unknown", "reason": "", "related_tags": []}
                for r in rec_items[:5]
            ],
            "correct": correct,
            "part_id": part_id,
            "current_tags": current_tags,
            "part_accuracy": part_acc,
            "reward": reward,
            "n_interactions": n_interactions,
        })

    return steps


print("Simulation function ready.")

In [ ]:
"""Cell 8 — Run simulation for all 5 selected students."""

case_study_data = {}  # cluster_id -> {uid, profile, steps}

for cid in sorted(selected_students.keys()):
    uid = selected_students[cid]
    user_test = test_df[test_df["user_id"] == uid]
    user_train = train_df[train_df["user_id"] == uid]

    logger.info(
        "Simulating cluster %d (%s): user=%s, n_test=%d",
        cid, pers._cluster_names.get(cid, "?"), uid, len(user_test),
    )
    t0 = time.time()
    steps = simulate_student(uid, user_test, n_steps=N_STEPS)
    elapsed = time.time() - t0

    # Build profile summary
    profile = user_profiles[uid].copy()
    profile["n_train_interactions"] = len(user_train)
    profile["n_test_interactions"] = len(user_test)
    profile["overall_accuracy"] = float(user_test["correct"].mean())
    if "part_id" in user_test.columns:
        dominant_part = int(user_test["part_id"].mode().iloc[0])
        profile["dominant_part"] = dominant_part
        profile["dominant_part_name"] = get_part_name(dominant_part)
    if steps:
        profile["final_theta"] = steps[-1]["theta"]
        profile["initial_theta"] = steps[0]["theta"]

    case_study_data[cid] = {
        "uid": uid,
        "profile": profile,
        "steps": steps,
        "time_s": round(elapsed, 1),
    }

    print(
        f"  Cluster {cid} ({profile['cluster_name']}): "
        f"{len(steps)} steps in {elapsed:.1f}s, "
        f"theta {profile.get('initial_theta', 0):.2f} -> {profile.get('final_theta', 0):.2f}"
    )

print(f"\nAll {len(case_study_data)} case studies simulated.")

In [ ]:
"""Cell 9 — Profile summary table."""

profile_rows = []
for cid in sorted(case_study_data.keys()):
    d = case_study_data[cid]
    p = d["profile"]
    profile_rows.append({
        "Cluster": cid,
        "Type": p.get("cluster_name", "?"),
        "User ID": d["uid"],
        "theta_initial": round(p.get("initial_theta", 0), 3),
        "theta_final": round(p.get("final_theta", 0), 3),
        "Delta_theta": round(p.get("final_theta", 0) - p.get("initial_theta", 0), 3),
        "Accuracy": round(p.get("overall_accuracy", 0), 3),
        "N_train": p.get("n_train_interactions", 0),
        "N_test": p.get("n_test_interactions", 0),
        "Dominant Part": p.get("dominant_part_name", "?"),
        "Dist to Centroid": round(p.get("distance_to_centroid", 0), 3),
    })

profile_table = pd.DataFrame(profile_rows)
display(profile_table)
profile_table.to_csv(RESULTS_DIR / "case_study_profiles.csv", index=False)
profile_table.to_latex(RESULTS_DIR / "case_study_profiles.tex", index=False, escape=False)
print("Saved: case_study_profiles.csv / .tex")

In [ ]:
"""Cell 10 — Main visualisation: 2x3 subplot per student.

Layout:
  (0,0): theta trajectory over 50 steps
  (0,1): Confidence class distribution (stacked bar per step window)
  (0,2): Strategy selection over time (TS weights as stacked area)
  (1,0): Accuracy per part (bar chart)
  (1,1): Gap tags -> prerequisite chain -> recommended lectures (table)
  (1,2): Summary metrics box
"""


def plot_case_study(cid, data, save_dir):
    """Generate 2x3 figure for a single student case study."""
    uid = data["uid"]
    profile = data["profile"]
    steps = data["steps"]
    cluster_name = profile.get("cluster_name", f"Cluster_{cid}")

    if not steps:
        print(f"  No steps for cluster {cid}, skipping.")
        return

    steps_df = pd.DataFrame(steps)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(
        f"Case Study: {cluster_name} (Cluster {cid}, User {uid})",
        fontsize=14, fontweight="bold", y=1.02,
    )

    # ---- (0,0): Theta trajectory ----
    ax = axes[0, 0]
    thetas = steps_df["theta"].values
    theta_se = steps_df["theta_se"].values
    x = np.arange(len(thetas))
    ax.plot(x, thetas, color="#1f77b4", linewidth=2, label="theta (IRT ability)")
    ax.fill_between(
        x, thetas - theta_se, thetas + theta_se,
        alpha=0.2, color="#1f77b4", label="+/- SE",
    )
    ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
    # Mark correct/incorrect
    correct_mask = steps_df["correct"].values
    ax.scatter(
        x[correct_mask], thetas[correct_mask],
        color="green", s=15, alpha=0.6, zorder=5, label="Correct",
    )
    ax.scatter(
        x[~correct_mask], thetas[~correct_mask],
        color="red", s=15, alpha=0.6, zorder=5, label="Incorrect",
    )
    ax.set_xlabel("Step")
    ax.set_ylabel("Theta (ability)")
    ax.set_title("(a) IRT Ability Trajectory")
    ax.legend(fontsize=8, loc="lower right")

    # ---- (0,1): Confidence class distribution (windowed) ----
    ax = axes[0, 1]
    window_size = 5
    n_windows = len(steps) // window_size

    # Collect all unique confidence class names
    all_conf_names = set()
    for s in steps:
        all_conf_names.update(s.get("conf_distribution", {}).keys())
    all_conf_names = sorted(all_conf_names)

    if all_conf_names and n_windows > 0:
        conf_matrix = np.zeros((n_windows, len(all_conf_names)))
        for w in range(n_windows):
            window_steps = steps[w * window_size : (w + 1) * window_size]
            combined = defaultdict(float)
            for s in window_steps:
                for name, frac in s.get("conf_distribution", {}).items():
                    combined[name] += frac
            total = sum(combined.values()) or 1.0
            for j, name in enumerate(all_conf_names):
                conf_matrix[w, j] = combined.get(name, 0) / total

        conf_colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(all_conf_names)))
        bottom = np.zeros(n_windows)
        x_w = np.arange(n_windows)
        for j, name in enumerate(all_conf_names):
            ax.bar(
                x_w, conf_matrix[:, j], bottom=bottom, width=0.8,
                label=name, color=conf_colors[j],
            )
            bottom += conf_matrix[:, j]

        ax.set_xticks(x_w)
        ax.set_xticklabels([f"{w*window_size}-{(w+1)*window_size-1}" for w in range(n_windows)],
                           rotation=45, fontsize=7)
        ax.legend(fontsize=7, loc="upper right")
    else:
        ax.text(0.5, 0.5, "No confidence data", ha="center", va="center",
                transform=ax.transAxes, fontsize=11)

    ax.set_xlabel("Step Window")
    ax.set_ylabel("Proportion")
    ax.set_title("(b) Confidence Class Distribution")
    ax.set_ylim(0, 1.05)

    # ---- (0,2): Strategy selection over time (TS weights stacked area) ----
    ax = axes[0, 2]
    strategies = ["knowledge_based", "content_based", "collaborative"]
    ts_matrix = np.zeros((len(steps), len(strategies)))
    for i, s in enumerate(steps):
        w = s.get("ts_weights", {})
        for j, strat in enumerate(strategies):
            ts_matrix[i, j] = w.get(strat, 0.0)

    # Normalise to sum=1 per step
    row_sums = ts_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    ts_norm = ts_matrix / row_sums

    ax.stackplot(
        np.arange(len(steps)),
        *[ts_norm[:, j] for j in range(len(strategies))],
        labels=[STRATEGY_LABELS[s] for s in strategies],
        colors=[STRATEGY_COLORS[s] for s in strategies],
        alpha=0.8,
    )

    # Overlay actual strategy choices as dots on top
    for i, s in enumerate(steps):
        strat = s.get("strategy", "knowledge_based")
        color = STRATEGY_COLORS.get(strat, "gray")
        ax.plot(i, 1.02, marker="v", color=color, markersize=4, alpha=0.7)

    ax.set_xlabel("Step")
    ax.set_ylabel("Normalised TS Weight")
    ax.set_title("(c) Thompson Sampling Strategy Weights")
    ax.set_ylim(0, 1.1)
    ax.legend(fontsize=8, loc="lower right")

    # ---- (1,0): Accuracy per part (bar chart) ----
    ax = axes[1, 0]
    final_part_acc = steps[-1].get("part_accuracy", {})
    if final_part_acc:
        parts = sorted(final_part_acc.keys())
        part_labels = [get_part_name(p) for p in parts]
        acc_vals = [final_part_acc[p] for p in parts]
        colors_bar = ["#2ca02c" if a >= 0.7 else "#ff7f0e" if a >= 0.5 else "#d62728"
                      for a in acc_vals]
        bars = ax.barh(part_labels, acc_vals, color=colors_bar, edgecolor="black", linewidth=0.5)
        ax.axvline(x=0.7, color="green", linestyle="--", alpha=0.5, label="Mastery (0.7)")
        ax.set_xlim(0, 1.0)
        for bar, val in zip(bars, acc_vals):
            ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height() / 2,
                    f"{val:.2f}", va="center", fontsize=8)
        ax.legend(fontsize=8)
    else:
        ax.text(0.5, 0.5, "No part data", ha="center", va="center",
                transform=ax.transAxes, fontsize=11)
    ax.set_xlabel("Accuracy")
    ax.set_title("(d) Accuracy per TOEIC Part")

    # ---- (1,1): Gap -> Prerequisite -> Lecture chain (text table) ----
    ax = axes[1, 1]
    ax.axis("off")

    # Aggregate gap/prerequisite/lecture data from last 10 steps
    recent_steps = steps[-10:]
    gap_prereq_chains = []
    seen_gaps = set()
    for s in recent_steps:
        for gtag_str, prereqs in s.get("prerequisite_gaps", {}).items():
            gtag = int(gtag_str)
            if gtag in seen_gaps:
                continue
            seen_gaps.add(gtag)
            # Find lectures recommended for this gap
            lectures_for_gap = []
            for r in s.get("recommended_items", []):
                if r.get("item_type") == "lecture" and gtag in r.get("related_tags", []):
                    lectures_for_gap.append(r["item_id"])
            gap_prereq_chains.append({
                "gap_tag": gtag,
                "prerequisites": prereqs[:3],
                "lectures": lectures_for_gap[:2],
            })
        if len(gap_prereq_chains) >= 5:
            break

    # Build table text
    table_header = "Gap Tag -> Prerequisites -> Lectures\n" + "-" * 45 + "\n"
    table_body = ""
    if gap_prereq_chains:
        for chain in gap_prereq_chains[:5]:
            prereq_str = ", ".join(str(p) for p in chain["prerequisites"]) or "none"
            lec_str = ", ".join(str(l) for l in chain["lectures"]) or "(direct practice)"
            table_body += f"Tag {chain['gap_tag']}\n"
            table_body += f"  prereqs: [{prereq_str}]\n"
            table_body += f"  lectures: [{lec_str}]\n\n"
    else:
        # Show gap tags and recommended items instead
        recent_gaps = []
        for s in recent_steps:
            recent_gaps.extend(s.get("gap_tags", []))
        recent_gaps = sorted(set(recent_gaps))[:8]
        if recent_gaps:
            table_body += f"Gap tags: {recent_gaps}\n\n"
        recent_recs = []
        for s in recent_steps[-3:]:
            for r in s.get("recommended_items", [])[:2]:
                recent_recs.append(f"{r['item_type']}: {r['item_id']}")
        if recent_recs:
            table_body += "Recent recommendations:\n"
            for rr in recent_recs[:5]:
                table_body += f"  {rr}\n"

    ax.text(
        0.05, 0.95, table_header + table_body,
        transform=ax.transAxes, fontsize=8, fontfamily="monospace",
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="lightyellow", alpha=0.8),
    )
    ax.set_title("(e) Recommendation Chain: Gap -> Prereqs -> Lectures")

    # ---- (1,2): Summary metrics box ----
    ax = axes[1, 2]
    ax.axis("off")

    # Compute summary stats
    strategies_used = pd.Series([s["strategy"] for s in steps]).value_counts()
    avg_reward = np.mean([s["reward"] for s in steps])
    n_gaps = len(set(g for s in steps for g in s.get("gap_tags", [])))
    rolling_acc = np.mean([s["correct"] for s in steps[-20:]])

    summary_text = (
        f"SUMMARY METRICS\n"
        f"{'=' * 30}\n"
        f"Cluster:         {cluster_name}\n"
        f"User ID:         {uid}\n"
        f"theta initial:   {profile.get('initial_theta', 0):.3f}\n"
        f"theta final:     {profile.get('final_theta', 0):.3f}\n"
        f"Delta theta:     {profile.get('final_theta', 0) - profile.get('initial_theta', 0):+.3f}\n"
        f"Overall acc:     {profile.get('overall_accuracy', 0):.1%}\n"
        f"Last-20 acc:     {rolling_acc:.1%}\n"
        f"Avg TS reward:   {avg_reward:.3f}\n"
        f"Unique gaps:     {n_gaps}\n"
        f"Dominant part:   {profile.get('dominant_part_name', '?')}\n"
        f"{'=' * 30}\n"
        f"Strategy distribution:\n"
    )
    for strat, count in strategies_used.items():
        pct = count / len(steps) * 100
        summary_text += f"  {STRATEGY_LABELS.get(strat, strat)}: {count} ({pct:.0f}%)\n"

    ax.text(
        0.05, 0.95, summary_text,
        transform=ax.transAxes, fontsize=9, fontfamily="monospace",
        verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.5", facecolor="lightcyan", alpha=0.8),
    )
    ax.set_title("(f) Summary")

    plt.tight_layout()
    fig_path = save_dir / f"case_study_cluster{cid}_{cluster_name.replace(' ', '_')}.png"
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    fig.savefig(fig_path.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fig_path.name}")


# Generate all case studies
for cid in sorted(case_study_data.keys()):
    print(f"\nPlotting cluster {cid} ({pers._cluster_names.get(cid, '?')})...")
    plot_case_study(cid, case_study_data[cid], RESULTS_DIR)

print("\nAll case study figures saved.")

In [ ]:
"""Cell 11 — Combined overview: all 5 students theta trajectories on one plot."""

fig, ax = plt.subplots(figsize=(14, 6))

cluster_colors = plt.cm.Set1(np.linspace(0, 1, max(len(case_study_data), 3)))

for i, cid in enumerate(sorted(case_study_data.keys())):
    d = case_study_data[cid]
    steps = d["steps"]
    if not steps:
        continue
    thetas = [s["theta"] for s in steps]
    label = f"C{cid}: {d['profile'].get('cluster_name', '?')} (u{d['uid']})"
    ax.plot(range(len(thetas)), thetas, linewidth=2, label=label,
            color=cluster_colors[i])

ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel("Step", fontsize=12)
ax.set_ylabel("Theta (IRT Ability)", fontsize=12)
ax.set_title("IRT Ability Trajectories: 5 Case Study Students",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="best")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "theta_trajectories_combined.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "theta_trajectories_combined.pdf", bbox_inches="tight")
plt.show()
print("Saved: theta_trajectories_combined.png / .pdf")

In [ ]:
"""Cell 12 — Combined TS weight evolution for all 5 students."""

fig, axes = plt.subplots(1, len(case_study_data), figsize=(4 * len(case_study_data), 4),
                          sharey=True)
if len(case_study_data) == 1:
    axes = [axes]

strategies = ["knowledge_based", "content_based", "collaborative"]

for ax, cid in zip(axes, sorted(case_study_data.keys())):
    d = case_study_data[cid]
    steps = d["steps"]
    if not steps:
        continue

    ts_matrix = np.zeros((len(steps), len(strategies)))
    for i, s in enumerate(steps):
        w = s.get("ts_weights", {})
        for j, strat in enumerate(strategies):
            ts_matrix[i, j] = w.get(strat, 0.0)

    row_sums = ts_matrix.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    ts_norm = ts_matrix / row_sums

    ax.stackplot(
        np.arange(len(steps)),
        *[ts_norm[:, j] for j in range(len(strategies))],
        labels=[STRATEGY_LABELS[s] for s in strategies],
        colors=[STRATEGY_COLORS[s] for s in strategies],
        alpha=0.8,
    )
    ax.set_title(f"C{cid}: {d['profile'].get('cluster_name', '?')}",
                 fontsize=10, fontweight="bold")
    ax.set_xlabel("Step")
    ax.set_ylim(0, 1)
    if ax == axes[0]:
        ax.set_ylabel("Normalised TS Weight")

axes[-1].legend(loc="center left", bbox_to_anchor=(1.05, 0.5), fontsize=8)

plt.suptitle("Thompson Sampling Weight Evolution by Cluster",
             fontsize=13, fontweight="bold", y=1.03)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "ts_weights_combined.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "ts_weights_combined.pdf", bbox_inches="tight")
plt.show()
print("Saved: ts_weights_combined.png / .pdf")

In [ ]:
"""Cell 13 — Interpretability analysis: per-cluster narrative.

For each cluster, generate a structured interpretation of WHY MARS
made the recommendations it did, tracing through the agent pipeline.
"""

interpretations = {}

for cid in sorted(case_study_data.keys()):
    d = case_study_data[cid]
    profile = d["profile"]
    steps = d["steps"]
    cluster_name = profile.get("cluster_name", f"Cluster_{cid}")

    if not steps:
        continue

    # Aggregate statistics
    thetas = [s["theta"] for s in steps]
    theta_trend = thetas[-1] - thetas[0]
    strategies_used = pd.Series([s["strategy"] for s in steps]).value_counts()
    dominant_strategy = strategies_used.index[0]
    all_gaps = set(g for s in steps for g in s.get("gap_tags", []))
    all_prereqs = set()
    for s in steps:
        for plist in s.get("prerequisite_gaps", {}).values():
            all_prereqs.update(plist)
    n_lectures_rec = sum(
        1 for s in steps
        for r in s.get("recommended_items", [])
        if r.get("item_type") == "lecture"
    )
    avg_reward = np.mean([s["reward"] for s in steps])

    # Final TS weights
    final_weights = steps[-1].get("ts_weights", {})

    interp = {
        "cluster_id": cid,
        "cluster_name": cluster_name,
        "user_id": d["uid"],
        "theta_trajectory": f"{thetas[0]:.3f} -> {thetas[-1]:.3f} (delta={theta_trend:+.3f})",
        "dominant_strategy": STRATEGY_LABELS.get(dominant_strategy, dominant_strategy),
        "strategy_distribution": {
            STRATEGY_LABELS.get(k, k): int(v)
            for k, v in strategies_used.items()
        },
        "n_unique_gaps": len(all_gaps),
        "n_prerequisite_tags": len(all_prereqs),
        "n_lectures_recommended": n_lectures_rec,
        "avg_reward": round(avg_reward, 4),
        "final_ts_weights": {k: round(v, 3) for k, v in final_weights.items()},
    }

    interpretations[cid] = interp

    # Print narrative
    print(f"\n{'='*60}")
    print(f"CLUSTER {cid}: {cluster_name}")
    print(f"{'='*60}")
    print(f"User {d['uid']} | Accuracy: {profile.get('overall_accuracy', 0):.1%}")
    print(f"Theta: {interp['theta_trajectory']}")
    print(f"\nWHY these recommendations:")
    print(f"  1. DiagnosticAgent estimated theta = {thetas[-1]:.3f}")
    print(f"     -> {'above' if thetas[-1] > 0 else 'below'} average ability")
    print(f"  2. KnowledgeGraphAgent identified {len(all_gaps)} gap tags")
    print(f"     -> {len(all_prereqs)} prerequisite tags need remediation")
    print(f"  3. RecommendationAgent selected {interp['dominant_strategy']} "
          f"most often ({strategies_used.iloc[0]}/{len(steps)} steps)")
    print(f"  4. Thompson Sampling final weights: {interp['final_ts_weights']}")
    print(f"  5. {n_lectures_rec} lecture recommendations across {len(steps)} steps")
    print(f"  6. Average reward (gap coverage): {avg_reward:.3f}")

In [ ]:
"""Cell 14 — Cross-cluster comparison: strategy adaptation.

Shows how MARS adapts its strategy mix differently for each learner type.
This is the key argument for multi-agent over monolithic.
"""

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel A: Strategy distribution by cluster (grouped bar) ---
ax = axes[0]
strategies = ["knowledge_based", "content_based", "collaborative"]
n_clusters = len(case_study_data)
x = np.arange(n_clusters)
width = 0.25

cluster_ids = sorted(case_study_data.keys())
for j, strat in enumerate(strategies):
    fracs = []
    for cid in cluster_ids:
        steps = case_study_data[cid]["steps"]
        if steps:
            strat_count = sum(1 for s in steps if s["strategy"] == strat)
            fracs.append(strat_count / len(steps))
        else:
            fracs.append(0)
    ax.bar(
        x + j * width, fracs, width,
        label=STRATEGY_LABELS[strat],
        color=STRATEGY_COLORS[strat],
        edgecolor="black", linewidth=0.5,
    )

ax.set_xticks(x + width)
ax.set_xticklabels(
    [f"C{cid}\n{case_study_data[cid]['profile'].get('cluster_name', '?')}"
     for cid in cluster_ids],
    fontsize=9,
)
ax.set_ylabel("Fraction of Steps")
ax.set_title("(a) Strategy Distribution by Cluster", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)

# --- Panel B: Reward vs theta delta (scatter) ---
ax2 = axes[1]
for i, cid in enumerate(cluster_ids):
    d = case_study_data[cid]
    steps = d["steps"]
    if not steps:
        continue
    theta_delta = steps[-1]["theta"] - steps[0]["theta"]
    avg_reward = np.mean([s["reward"] for s in steps])
    cluster_name = d["profile"].get("cluster_name", "?")
    ax2.scatter(
        theta_delta, avg_reward,
        s=200, color=cluster_colors[i], edgecolors="black", linewidth=1.5,
        zorder=5,
    )
    ax2.annotate(
        f"C{cid}: {cluster_name}",
        (theta_delta, avg_reward),
        textcoords="offset points", xytext=(10, 5),
        fontsize=9, fontweight="bold",
    )

ax2.set_xlabel("Delta Theta (ability change)", fontsize=11)
ax2.set_ylabel("Average Reward (gap coverage)", fontsize=11)
ax2.set_title("(b) Ability Change vs Recommendation Reward", fontweight="bold")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "cross_cluster_comparison.png", dpi=300, bbox_inches="tight")
plt.savefig(RESULTS_DIR / "cross_cluster_comparison.pdf", bbox_inches="tight")
plt.show()
print("Saved: cross_cluster_comparison.png / .pdf")

## Key Findings (expected)

1. **Diverse theta trajectories** -- each cluster shows a qualitatively different
   ability trajectory, confirming that MARS handles heterogeneous learners.

2. **Strategy adaptation** -- Thompson Sampling converges to *different* strategy
   distributions per cluster:
   - *Fast Learners*: shift toward Content-Based (need novel content, not remediation)
   - *Struggling*: dominated by Knowledge-Based (gap-filling via prerequisites)
   - *Improving*: balanced KB/CB mix (building on progress)
   - *Cramming*: higher Collaborative weight (benefit from peer patterns)
   - *Methodical/Steady*: stable KB-dominant with some CB exploration

3. **Interpretable chains** -- for every recommendation we can trace:
   *DiagnosticAgent (theta)* -> *KGAgent (gap tags)* -> *KGAgent (prerequisites)* ->
   *RecommendationAgent (lecture/question)* -> *TS (strategy weight update)*

4. **Multi-agent advantage** -- a monolithic model would produce the same opaque
   embedding for all learners. MARS makes the *why* explicit at every step.

5. **TS reward feedback loop** -- the reward signal from ground-truth gaps feeds
   back into Thompson Sampling posteriors, enabling online adaptation without retraining.

In [ ]:
"""Cell 15 — Save all results."""

# Save interpretations
with open(RESULTS_DIR / "case_study_interpretations.json", "w") as f:
    json.dump(interpretations, f, indent=2, default=str)

# Save raw step data (for reproducibility)
raw_data = {}
for cid in sorted(case_study_data.keys()):
    d = case_study_data[cid]
    raw_data[str(cid)] = {
        "uid": str(d["uid"]),
        "profile": {
            k: (v.tolist() if isinstance(v, np.ndarray) else v)
            for k, v in d["profile"].items()
        },
        "steps": d["steps"],
        "time_s": d["time_s"],
    }

with open(RESULTS_DIR / "case_study_raw.json", "w") as f:
    json.dump(raw_data, f, indent=2, default=str)

# Save metadata
meta = {
    "experiment": "MARS Case Studies: Interpretability via Individual Learner Trajectories",
    "seed": SEED,
    "n_steps": N_STEPS,
    "n_students": len(case_study_data),
    "min_test_interactions": MIN_TEST_INTERACTIONS,
    "clusters": {
        str(cid): {
            "name": case_study_data[cid]["profile"].get("cluster_name", "?"),
            "user_id": str(case_study_data[cid]["uid"]),
        }
        for cid in sorted(case_study_data.keys())
    },
    "output_files": [
        "case_study_profiles.csv",
        "case_study_profiles.tex",
        "case_study_interpretations.json",
        "case_study_raw.json",
        "theta_trajectories_combined.png",
        "ts_weights_combined.png",
        "cross_cluster_comparison.png",
    ] + [
        f"case_study_cluster{cid}_{case_study_data[cid]['profile'].get('cluster_name', 'X').replace(' ', '_')}.png"
        for cid in sorted(case_study_data.keys())
    ],
}

with open(RESULTS_DIR / "case_study_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print("All results saved.")
print(f"Results directory: {RESULTS_DIR}")
for fname in meta["output_files"]:
    full = RESULTS_DIR / fname
    status = "OK" if full.exists() else "(pending execution)"
    print(f"  {fname} ... {status}")